# 03 — Feature engineering, temporal validation, and training

This notebook turns the municipality-month table from `02` into the model inputs,
splits the series in time, and trains the six predictors compared in the paper.

**Six predictors.** Two baselines and four learned models. *Persistence* predicts
that next month repeats this month. *Climatology* predicts each municipality's
historical rate for that calendar month. The learned models are logistic
regression, random forest, XGBoost and LightGBM. All of them see the same
features, the same folds and the same labels, so the comparison is like for like.

**Expanding-window validation.** To score 2022 the model trains on everything
whose target falls before 2022; to score 2023, everything before 2023; and so on.
Nothing from the future ever enters training.

**Four leakage guards** are described where they are applied: the fold boundary
follows the month being predicted rather than the month observed; rolling windows
are built with a group-wise transform so no window crosses between
municipalities; every rolling feature is shifted before it is aggregated; and no
feature derives from a label whose threshold was fitted over the full series.

Outputs land in `data/train/`: one metrics file and one prediction file per
disease, definition and horizon, plus fitted models for the subset listed in
`SAVE_ALL_MODELS`.


## Section 0 — Setup: load master + labels

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd().parent
MASTER_DIR = PROJECT_ROOT / "data" / "master"
OUT_DIR = PROJECT_ROOT / "data" / "train"
OUT_DIR.mkdir(parents=True, exist_ok=True)

X_master = pd.read_parquet(MASTER_DIR / "municipio_mes.parquet")
y_labels = pd.read_parquet(MASTER_DIR / "labels.parquet")

df = X_master.merge(y_labels, on=["cod_ibge", "ano", "mes"], how="left")
df = df.sort_values(["cod_ibge", "ano", "mes"]).reset_index(drop=True)
print(f"Merged: {df.shape}")

Merged: (89010, 99)


## Section 1 — Build features (leakage-safe)

Every feature at *t* is computed from data available at *t* and no later. Three
rules do the work, and each exists because its absence is a specific way to leak:

- **Lags use `shift(k)`** within the municipality, so a value never crosses from
  one municipality into another.
- **Rolling means and trends use `shift(1)` before the window**, so the window
  ends at *t-1* and never contains the month being described. They also run
  through `transform`, which keeps the window inside the municipality — applying
  `.rolling()` to a flat shifted series pulls values across that boundary.
- **The lagged endemic-channel label is deliberately excluded.** It reads as a
  harmless "was there an outbreak last month", but its threshold was fitted over
  the whole series, so the value encodes years that come after the row. The
  reasoning and the measurement are in the comment where it would have been built.

One naming quirk of this master to keep in mind: yellow-fever cases use the `fa_`
prefix while its SIH hospitalizations use `febre_amarela_`.

In [2]:
DISEASES = ["dengue", "zika", "chikungunya", "fa"]
CLIMATE_VARS = ["t2m", "prectotcorr", "rh2m", "ws10m", "ps"]
LAGS_CASES = [1, 2, 3, 6, 12]
ROLL_WINDOWS = [3, 6]

# Fill case NaN -> 0 (no notification = zero inside the grid) BEFORE any grouping,
# and add incidence + past-outbreak columns to df, so a single groupby sees them all.
for d in DISEASES:
    df[f"{d}_casos"] = df[f"{d}_casos"].fillna(0)
    df[f"{d}_incid"] = df[f"{d}_casos"] / df["pop_residente"] * 100_000

# NOTE — the lagged endemic-channel label is deliberately NOT used as a feature.
# It looks harmless (a past outbreak is observable at the time), but the canal
# label's threshold is median + 1.96*sigma over the NON-EPIDEMIC years of the
# whole series, and for dengue that set includes 2022 and 2023 — test years.
# Recomputing the label with a baseline restricted to years before 2022 flips
# 2,722 of 54,180 pre-test labels (5.0%), across 525 of the 645 municipalities:
# 4,200 pre-test outbreaks become 6,494. So the value a model would see in 2018
# encodes how bad 2022-2023 turned out. That is lookahead, and it goes out.
# The label itself stays as it is: as a TARGET the fixed window is a documented,
# defensible choice — it is only its use as an INPUT that leaks.

# SIH hospitalizations: fill 0; yellow fever uses 'febre_amarela' prefix
SIH_COL = {"dengue": "dengue_internacoes_sih", "zika": "zika_internacoes_sih",
           "chikungunya": "chikungunya_internacoes_sih", "fa": "febre_amarela_internacoes_sih"}
for d in DISEASES:
    if SIH_COL[d] in df.columns:
        df[SIH_COL[d]] = df[SIH_COL[d]].fillna(0)

# ONE groupby, created AFTER all df columns exist -> no stale-groupby leakage
g = df.groupby("cod_ibge", group_keys=False)

def lag(col, k):
    return g[col].shift(k)

def roll_mean(col, w):
    # Two guards, both necessary.
    # shift(1) first: the window ends at t-1 and never includes t.
    # transform: the rolling window must stay INSIDE the municipality. Calling
    # .rolling() on g[col].shift(1) rolls over the FLAT series and pulls values
    # across the boundary between municipalities — measured on this base: 1,148
    # rows, every one of them the first or second month of a municipality.
    return g[col].transform(lambda s: s.shift(1).rolling(w, min_periods=1).mean())

def trend3(col):
    def slope(x):
        y = x.values
        m = ~np.isnan(y)
        if m.sum() < 2:
            return np.nan
        t = np.arange(len(y))
        return np.polyfit(t[m], y[m], 1)[0]
    # transform, for the same reason as roll_mean: the window must not reach
    # back into the previous municipality.
    return g[col].transform(
        lambda s: s.shift(1).rolling(3, min_periods=2).apply(slope, raw=False))

feats = df[["cod_ibge", "ano", "mes"]].copy()

# --- per-disease: case lags/rolls, incidence lags/rolls, trend, past outbreaks ---
for d in DISEASES:
    for k in LAGS_CASES:
        feats[f"{d}_casos_lag{k}"] = lag(f"{d}_casos", k)
    for w in ROLL_WINDOWS:
        feats[f"{d}_casos_roll{w}"] = roll_mean(f"{d}_casos", w)
    for k in [1, 2, 3]:
        feats[f"{d}_incid_lag{k}"] = lag(f"{d}_incid", k)
    for w in ROLL_WINDOWS:
        feats[f"{d}_incid_roll{w}"] = roll_mean(f"{d}_incid", w)
    feats[f"{d}_casos_trend3"] = trend3(f"{d}_casos")

# --- climate: current month + lag1/lag2 + roll3 ---
for v in CLIMATE_VARS:
    feats[v] = df[v].values
    feats[f"{v}_lag1"] = lag(v, 1)
    feats[f"{v}_lag2"] = lag(v, 2)
    feats[f"{v}_roll3"] = roll_mean(v, 3)

# --- seasonality (cyclic) ---
feats["mes_sin"] = np.sin(2 * np.pi * feats["mes"] / 12)
feats["mes_cos"] = np.cos(2 * np.pi * feats["mes"] / 12)

# --- operational-delay sources: lag1 ---
for d in DISEASES:
    if SIH_COL[d] in df.columns:
        feats[f"{SIH_COL[d]}_lag1"] = lag(SIH_COL[d], 1)
for d in ["dengue", "zika", "chikungunya"]:
    for stem in (f"{d}_latencia_mediana", f"{d}_latencia_p90"):
        if stem in df.columns:
            feats[f"{stem}_lag1"] = lag(stem, 1)
for c in ("esf_cobertura_pct", "esf_qt_equipes"):
    if c in df.columns:
        feats[f"{c}_lag1"] = lag(c, 1)

# --- static / annual: pass through ---
STATIC_ANNUAL = [
    "pop_residente", "pib_per_capita", "densidade_hab_km2", "area_km2",
    "gini_2010", "idhm_2010", "rmm_por_100k_nv",
    "num_favelas", "pop_favelas", "cob_vac_fa_pct", "pendulares_saem_trabalho",
    "iag0001_atend_agua_pct", "ies0001_atend_esgoto_pct", "ies0007_esgoto_tratado_pct",
    "iap0001_vias_pavimentadas_pct",
    "pct_floresta", "pct_agricultura", "pct_nao_vegetado", "pct_agua", "pct_natural_nao_florestal",
    "msau28_pacs", "msau541_vig_sanitaria", "msau542_vig_epidemiologica", "msau543_controle_endemias",
    "mgrd01_seca", "mgrd06_alagamento", "mgrd07_erosao", "mgrd08_enchente_gradual",
    "mgrd11_enxurrada", "mgrd14_deslizamento", "mgrd201_mapeamento_risco", "mmam2612_moradia_risco",
    "esf_pop_referencia",
]
for c in STATIC_ANNUAL:
    if c in df.columns:
        feats[c] = df[c].values

# --- categoricals -> one-hot ---
onehot_frames = []
for cat in ("capag", "esf_metodologia"):
    if cat in df.columns:
        onehot_frames.append(pd.get_dummies(df[cat], prefix=cat, dummy_na=False).astype(int))
if onehot_frames:
    feats = pd.concat([feats.reset_index(drop=True)] + [f.reset_index(drop=True) for f in onehot_frames], axis=1)

print(f"Features: {feats.shape}")
all_nan = [c for c in feats.columns if feats[c].isna().all()]
print("All-NaN columns:", all_nan if all_nan else "none")
feats.head(3)

Features: (89010, 128)
All-NaN columns: none


/tmp/ipykernel_8300/3029281297.py:110: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  feats[c] = df[c].values
/tmp/ipykernel_8300/3029281297.py:110: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  feats[c] = df[c].values
/tmp/ipykernel_8300/3029281297.py:110: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`


,cod_ibge,ano,mes,dengue_casos_lag1,dengue_casos_lag2,dengue_casos_lag3,dengue_casos_lag6,dengue_casos_lag12,dengue_casos_roll3,dengue_casos_roll6,...,mgrd14_deslizamento,mgrd201_mapeamento_risco,mmam2612_moradia_risco,esf_pop_referencia,capag_A,capag_B,capag_C,capag_D,esf_metodologia_AB,esf_metodologia_APS
0,3500105,2015,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,None,False,None,35001.0,0,0,0,0,1,0
1,3500105,2015,2,9.0,NaN,NaN,NaN,NaN,9.0,9.0,...,None,False,None,35001.0,0,0,0,0,1,0
2,3500105,2015,3,48.0,9.0,NaN,NaN,NaN,28.5,28.5,...,None,False,None,35001.0,0,0,0,0,1,0


In [3]:
# Convert boolean-like columns (True/False, possibly with NaN) to numeric float.
# XGBoost accepts float with NaN (treats NaN as missing), but rejects object dtype.
for c in feats.columns:
    if feats[c].dtype == "object" or feats[c].dtype == "bool":
        # map True/False to 1.0/0.0, keep NaN as NaN
        feats[c] = feats[c].map({True: 1.0, False: 0.0}).astype("float32")

# --- Annual sources with a publication lag ------------------------------
# Seven columns come from sources that publish once a year and late. IBGE
# municipal GDP runs about two years behind; MapBiomas land cover had not yet
# released the current year when this was collected. For the months the source
# has not reached, `02_master` leaves a NaN on purpose: there, a gap means "not
# published", and the master table is meant to record what each source said,
# not to guess.
#
# The guess belongs here instead, because this is where modelling decisions are
# made. Left alone, those NaNs reach the training call and `fillna(0)` turns
# them into zero, so a model reads a municipality as having no forest, no water
# and a GDP per capita of zero. That is not a missing value, it is a wrong one,
# and it is the kind of error a tree model will happily split on.
#
# These variables barely move: median year-on-year change is 0.08 for forest
# cover on a typical value of 8.74, and 0.04 for surface water on 0.45. Carrying
# the last published year forward errs by about one per cent; zeroing errs by a
# hundred.
#
# Forward only, never backward. All seven have data from 2015, the first year of
# the grid, so no row ever needs a later value carried back and the question of
# a future value reaching a past row does not arise. Grouped by municipality,
# for the same reason every rolling window in this notebook is grouped: without
# it the last month of one municipality would feed the first month of the next.
FONTES_ANUAIS_DEFASADAS = [
    # Fontes anuais cuja publicação atrasa: o valor existe, só não saiu ainda.
    "pib_per_capita", "densidade_hab_km2", "pct_floresta", "pct_agricultura",
    "pct_nao_vegetado", "pct_agua", "pct_natural_nao_florestal",
    # Séries mensais/anuais com falha de reporte pontual: mesma lógica.
    "esf_cobertura_pct_lag1", "esf_qt_equipes_lag1", "esf_pop_referencia",
    "cob_vac_fa_pct",
]
_presentes = [c for c in FONTES_ANUAIS_DEFASADAS if c in feats.columns]
feats = feats.sort_values(["cod_ibge", "ano", "mes"]).reset_index(drop=True)
_antes = feats[_presentes].isna().sum().sum()
for c in _presentes:
    feats[c] = feats.groupby("cod_ibge")[c].ffill()
_depois = feats[_presentes].isna().sum().sum()
print(f"Annual sources carried forward: {len(_presentes)} columns, "
      f"{_antes - _depois:,} cells filled, {_depois:,} still missing")

# Everything else keeps the treatment it had. A NaN in a case, death or
# hospitalisation column means no notification that month, which is information
# about the epidemic rather than a gap: carrying last month's count forward
# would fabricate an outbreak. Notification-latency columns are NaN for the same
# reason and stay that way; the training call fills them with zero, which is
# imperfect but harmless, because those are months with no cases and the count
# columns already tell the model so. After this step about 6.95% of the feature
# matrix is still NaN and still becomes zero downstream.

# --- Política de valores ausentes -------------------------------------------
# Decidida coluna a coluna, não por reflexo. Três regras, nesta ordem:
#
# 1. PROPAGAR (11 colunas, acima). O valor existe e só está atrasado ou faltou
#    num mês de reporte. Carregar o último conhecido erra por cerca de um por
#    cento; zerar erra por cem.
#
# 2. MANTER NULO (as de baixo). Ausência que não é zero e não tem passado de
#    onde vir. Os modelos de árvore aprendem uma direção para o ausente e a
#    regressão logística imputa pela mediana no próprio pipeline dela.
#
# 3. ZERO (todo o resto, 80 colunas). Casos, óbitos, internações e favelas,
#    onde nulo significa "não houve" e zero é o valor correto; latência de
#    notificação, nula justamente nos meses sem caso; e MUNIC e saneamento,
#    onde o que resta é município que a pesquisa nunca cobriu.
#
# Medimos a alternativa de entregar todos os nulos aos modelos: as três
# árvores pioram (LightGBM perde 0.0057 de AUPRC) e só a logística melhora.
# A maioria dos nossos nulos significa zero de verdade, então preservá-los em
# massa tira informação em vez de acrescentar.
COLUNAS_MANTER_NULO = [
    # Três municípios não constam da fonte do IDH. Zero num índice de 0 a 1 os
    # colocaria como os mais miseráveis do estado, o que é falso.
    "idhm_2010",
    # Médias móveis de clima no primeiro mês de cada município: o .shift(1) não
    # tem passado de onde tirar média. Zero significaria 0 °C num estado cuja
    # média é 23,2.
    "t2m_roll3", "prectotcorr_roll3", "rh2m_roll3", "ws10m_roll3", "ps_roll3",
]


def preencher(X):
    """Aplica a política de ausentes a uma matriz de features.

    Chamada em todos os pontos que ajustam ou pontuam um modelo, nos dois
    notebooks, para que treino, avaliação e produção nunca divirjam. Definida
    aqui, e não junto do treino, porque o 04_results reexecuta esta célula.
    """
    manter = [c for c in COLUNAS_MANTER_NULO if c in X.columns]
    if not manter:
        return X.fillna(0)
    guardado = X[manter].copy()
    X = X.fillna(0)
    X[manter] = guardado
    return X

# confirm no more object columns
obj_cols = [c for c in feats.columns if feats[c].dtype == "object"]
print("Object columns remaining:", obj_cols if obj_cols else "none")
print("dtypes sample:", feats[["mgrd01_seca", "msau28_pacs", "mmam2612_moradia_risco"]].dtypes.to_dict())

Annual sources carried forward: 11 columns, 54,483 cells filled, 1,842 still missing
Object columns remaining: none
dtypes sample: {'mgrd01_seca': dtype('float32'), 'msau28_pacs': dtype('float32'), 'mmam2612_moradia_risco': dtype('float32')}


## Section 2 — Shifted target per horizon h

In [4]:
def build_target(labels_df, disease, definition, h):
    # outbreak at t+h, per municipality; last h months become NaN (dropped later)
    col = f"{disease}_{definition}"
    return (labels_df.sort_values(["cod_ibge", "ano", "mes"])
                     .groupby("cod_ibge")[col].shift(-h))

_t = build_target(y_labels, "dengue", "inc100", 1)
print(f"dengue inc100 h=1: {_t.notna().sum():,} non-null, {int(_t.sum()):,} outbreaks")

dengue inc100 h=1: 88,365 non-null, 14,116 outbreaks


## Section 3 — Expanding-window temporal split

In [5]:
TEST_YEARS = [2022, 2023, 2024]

def add_target_year(frame, h):
    """Year of the outbreak being predicted, h months ahead of the observation.

    The fold split keys on the year of the TARGET, not of the observation: a row
    belongs in training only if the outbreak it predicts happened before the test
    year. The obvious `ano + (mes == 12)` is correct ONLY for h == 1, because only
    December crosses the year boundary one month ahead. At h == 6, months 7 to 11
    also land in the next year — and with the naive formula those rows keep the
    previous year, so they train on labels from the test period while rows late in
    the test year are scored against outcomes from the year after it.
    """
    frame = frame.copy()
    total = frame["ano"] * 12 + (frame["mes"] - 1) + h
    frame["target_year"] = total // 12
    return frame

def expanding_folds(frame, test_years=TEST_YEARS):
    for ty in test_years:
        yield (ty,
               frame.index[frame["target_year"] < ty],
               frame.index[frame["target_year"] == ty])

# Fold sizes shift with the horizon, precisely because the target year does.
for h_demo in (1, 6):
    print(f"h={h_demo}:")
    for ty, tr, te in expanding_folds(add_target_year(feats.copy(), h_demo)):
        print(f"  test {ty}: train={len(tr):,}  test={len(te):,}")

h=1:


  test 2022: train=53,535  test=7,740
  test 2023: train=61,275  test=7,740
  test 2024: train=69,015  test=7,740
h=6:


  test 2022: train=50,310  test=7,740
  test 2023: train=58,050  test=7,740
  test 2024: train=65,790  test=7,740


## Section 4 — Validate one combination before scaling

Train dengue/inc100/RF/h=1 on the last fold before running the whole grid, as a
smoke test on the plumbing rather than a result.

What to look for is the *order of magnitude*, not a target number. AUROC in the
high 0.8s to low 0.9s on this task is what a sound pipeline produces. **A value
near 0.99 means something leaked** — that is the signal to stop and find it, not
to celebrate.

In [6]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score, average_precision_score

disease, definition, h = "dengue", "inc100", 1

work = feats.copy()
work["target"] = build_target(y_labels, disease, definition, h).values
work = add_target_year(work, h).dropna(subset=["target"])
work["target"] = work["target"].astype(int)

NON_FEATURES = ["cod_ibge", "ano", "mes", "target", "target_year"]
feat_cols = [c for c in work.columns if c not in NON_FEATURES]

ty = 2024
tr = work.index[work["target_year"] < ty]
te = work.index[work["target_year"] == ty]
X_train, y_train = preencher(work.loc[tr, feat_cols]), work.loc[tr, "target"]
X_test,  y_test  = preencher(work.loc[te, feat_cols]), work.loc[te, "target"]

print(f"Train {X_train.shape}, pos {100*y_train.mean():.1f}%  |  Test {X_test.shape}, pos {100*y_test.mean():.1f}%")

rf = RandomForestClassifier(n_estimators=200, class_weight="balanced", n_jobs=-1, random_state=42)
rf.fit(X_train, y_train)
proba = rf.predict_proba(X_test)[:, 1]
print(f"\ndengue inc100 h=1, test 2024:")
auroc = roc_auc_score(y_test, proba)
print(f"  AUROC {auroc:.3f}")
print(f"  AUPRC {average_precision_score(y_test, proba):.3f}   "
      f"(chance level = prevalence {y_test.mean():.3f})")
if auroc > 0.98:
    print("\n  WARNING: AUROC above 0.98 on this task means something leaked."
          "\n  Stop and audit the features before running the full grid.")

Train (69015, 125), pos 12.3%  |  Test (7740, 125), pos 39.0%



dengue inc100 h=1, test 2024:
  AUROC 0.880
  AUPRC 0.824   (chance level = prevalence 0.390)


In [7]:
# Compare our feature set against the old pipeline's expected features
NON_FEATURES = ["cod_ibge", "ano", "mes"]
our_feats = set(c for c in feats.columns if c not in NON_FEATURES)
print(f"Our feature count: {len(our_feats)}")

# list our features grouped by type, to eyeball what we have
import re
print("\nOur climate features:", sorted([c for c in our_feats if any(v in c for v in ['t2m','prectotcorr','rh2m','ws10m','ps'])]))
print("\nOur one-hot (capag/esf_metodologia):", sorted([c for c in our_feats if c.startswith(('capag','esf_metodologia'))]))
print("\nCoordinates present?", [c for c in our_feats if c in ('lat','lon','dist_estacao_km')])

Our feature count: 125

Our climate features: ['prectotcorr', 'prectotcorr_lag1', 'prectotcorr_lag2', 'prectotcorr_roll3', 'ps', 'ps_lag1', 'ps_lag2', 'ps_roll3', 'rh2m', 'rh2m_lag1', 'rh2m_lag2', 'rh2m_roll3', 't2m', 't2m_lag1', 't2m_lag2', 't2m_roll3', 'ws10m', 'ws10m_lag1', 'ws10m_lag2', 'ws10m_roll3']

Our one-hot (capag/esf_metodologia): ['capag_A', 'capag_B', 'capag_C', 'capag_D', 'esf_metodologia_AB', 'esf_metodologia_APS']

Coordinates present? []


## Section 5 — Training loop (paper subset, checkpointed)

Filled together after Section 4 validates. Trains all diseases × definitions × tree models + baselines × 6 horizons × 3 folds, saving predictions and metrics to disk as each finishes.

In [8]:
import time
import joblib
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score, average_precision_score
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
import lightgbm as lgb
import xgboost as xgb

# ===============================================================
# Section 5 - Fold evaluation
#   Full grid: 6 models x 4 diseases x 4 definitions x 6 horizons x 3 folds.
#   The six are the four learned models plus the persistence and climatology
#   baselines. Saves metrics (results/), per-sample predictions (predictions/)
#   and every fitted model (models/).
#   Dengue runs first, so the numbers that matter most land on disk early and
#   04 can be started while the rarer diseases finish.
#   Checkpointed: re-running skips combos already complete on disk.
# ===============================================================
RESULTS_DIR = OUT_DIR / "results"
PRED_DIR    = OUT_DIR / "predictions"
MODELS_DIR  = OUT_DIR / "models"
for d in (RESULTS_DIR, PRED_DIR, MODELS_DIR):
    d.mkdir(parents=True, exist_ok=True)

FEAT_COLS_EXCLUDE = ["cod_ibge", "ano", "mes", "target", "target_year"]

# Fitted artifacts are written for every combination. The web application built
# on this repository lets a user pick any disease, definition, horizon and model
# and explains an individual prediction with SHAP, which needs that exact model
# on disk. The cost is roughly 1.5 GB across ~1,080 files; the whole tree is
# gitignored and is regenerated by re-running this notebook.
SAVE_ALL_MODELS = True

def model_factories():
    def rf():   return RandomForestClassifier(n_estimators=200, min_samples_leaf=20,
                                              class_weight="balanced", n_jobs=-1, random_state=42)
    def lgbm(): return lgb.LGBMClassifier(n_estimators=300, num_leaves=63, learning_rate=0.05,
                                          class_weight="balanced", n_jobs=-1, verbose=-1, random_state=42)
    def xgbm(): return xgb.XGBClassifier(n_estimators=300, max_depth=6, learning_rate=0.05,
                                         subsample=0.8, colsample_bytree=0.8, tree_method="hist",
                                         n_jobs=-1, eval_metric="aucpr", random_state=42)
    def logreg(): return Pipeline([("imp", SimpleImputer(strategy="median")),
                                   ("sc", StandardScaler()),
                                   ("clf", LogisticRegression(class_weight="balanced",
                                                              max_iter=2000, random_state=42))])
    return {"rf": rf, "lgbm": lgbm, "xgb": xgbm, "logreg": logreg}

def baseline_persistence(work_te, disease, definition):
    col = f"{disease}_{definition}"
    return work_te.merge(y_labels[["cod_ibge","ano","mes",col]],
                         on=["cod_ibge","ano","mes"], how="left")[col].fillna(0).values.astype(float)

def baseline_climatology(work_tr, work_te, disease, definition):
    col = f"{disease}_{definition}"
    tr = work_tr.merge(y_labels[["cod_ibge","ano","mes",col]],
                       on=["cod_ibge","ano","mes"], how="left")
    freq = tr.groupby(["cod_ibge","mes"])[col].mean().rename("clim")
    return work_te.merge(freq, on=["cod_ibge","mes"], how="left")["clim"].fillna(0).values

def prepare_work(disease, definition, h):
    work = feats.copy()
    work["target"] = build_target(y_labels, disease, definition, h).values
    work = add_target_year(work, h).dropna(subset=["target"])
    work["target"] = work["target"].astype(int)
    feat_cols = [c for c in work.columns if c not in FEAT_COLS_EXCLUDE]
    return work, feat_cols

def score(yte, proba):
    if yte.nunique() < 2:
        return float("nan"), float("nan")
    return roc_auc_score(yte, proba), average_precision_score(yte, proba)

def run_combo(disease, definition, h, save_models=True):
    work, feat_cols = prepare_work(disease, definition, h)
    factories = model_factories()
    rows, preds = [], []
    for ty, tr_idx, te_idx in expanding_folds(work):
        if len(te_idx) == 0 or work.loc[tr_idx, "target"].sum() == 0:
            continue
        Xtr, ytr = preencher(work.loc[tr_idx, feat_cols]), work.loc[tr_idx, "target"]
        Xte, yte = preencher(work.loc[te_idx, feat_cols]), work.loc[te_idx, "target"]
        meta_te  = work.loc[te_idx, ["cod_ibge","ano","mes"]]
        probas = {}
        for name, factory in factories.items():
            try:
                mdl = factory()
                if name == "xgb":
                    npos = int(ytr.sum()); nneg = len(ytr)-npos
                    mdl.set_params(scale_pos_weight=(nneg/max(npos,1)))
                mdl.fit(Xtr, ytr)
                probas[name] = mdl.predict_proba(Xte)[:,1]
                if save_models:
                    fn = MODELS_DIR / f"{disease}_{definition}_h{h}_{name}_{ty}.joblib"
                    joblib.dump(mdl, fn, compress=3)
            except Exception as e:
                print(f"      [skip {name} {ty}] {e}")
        probas["persistence"] = baseline_persistence(work.loc[te_idx], disease, definition)
        probas["climatology"] = baseline_climatology(work.loc[tr_idx], work.loc[te_idx], disease, definition)
        for name, proba in probas.items():
            au, ap = score(yte, proba)
            rows.append({"disease":disease,"definition":definition,"h":h,"test_year":ty,
                         "model":name,"auroc":au,"auprc":ap,"n_test":len(yte),"n_pos":int(yte.sum())})
            p = meta_te.copy()
            p["disease"]=disease; p["definition"]=definition; p["h"]=h
            p["model"]=name; p["test_year"]=ty; p["y_true"]=yte.values; p["proba"]=proba
            preds.append(p)
    return rows, preds

# The fold evaluation answers "how good is this method" and its result does not
# move when a new month arrives: fold `ty` always trains on target_year < ty.
# A monthly production run can therefore skip it and keep what is on disk,
# which cuts about fifty minutes. Set it to True to re-evaluate the method.
RODAR_FOLDS = True

DISEASES_ORDER = ["dengue", "zika", "chikungunya", "fa"]
DEFS_ORDER     = ["inc100", "canal", "inc300", "zscore"]
HORIZONS       = [1, 2, 3, 4, 5, 6]
combos = [(d, defn, h) for d in DISEASES_ORDER for defn in DEFS_ORDER for h in HORIZONS]

if not RODAR_FOLDS:
    print("RODAR_FOLDS is False: keeping the fold results already on disk.")

t0 = time.time()
for i, (disease, definition, h) in enumerate(combos if RODAR_FOLDS else [], 1):
    ckpt = RESULTS_DIR / f"{disease}_{definition}_h{h}.parquet"
    # A combination counts as done only if its models are on disk too. Without
    # this, a run made before SAVE_ALL_MODELS would be skipped and would stay
    # unexplainable in the application.
    tem_modelos = any(MODELS_DIR.glob(f"{disease}_{definition}_h{h}_*.joblib"))
    if ckpt.exists() and (tem_modelos or not SAVE_ALL_MODELS):
        print(f"[{i}/{len(combos)}] {disease}/{definition}/h{h}: exists, skip")
        continue
    keep_models = SAVE_ALL_MODELS
    rows, preds = run_combo(disease, definition, h, save_models=keep_models)
    if rows:
        pd.DataFrame(rows).to_parquet(ckpt)
        pd.concat(preds, ignore_index=True).to_parquet(PRED_DIR / f"{disease}_{definition}_h{h}.parquet")
    print(f"[{i}/{len(combos)}] {disease}/{definition}/h{h}: {len(rows)} results"
          f"{' +models' if keep_models else ''} ({time.time()-t0:.0f}s)")

print(f"\nAll combos done in {time.time()-t0:.0f}s")
print(f"Models saved: {len(list(MODELS_DIR.glob('*.joblib')))}")

[1/96] dengue/inc100/h1: 18 results +models (51s)


[2/96] dengue/inc100/h2: 18 results +models (101s)


[3/96] dengue/inc100/h3: 18 results +models (145s)


[4/96] dengue/inc100/h4: 18 results +models (189s)


[5/96] dengue/inc100/h5: 18 results +models (233s)


[6/96] dengue/inc100/h6: 18 results +models (279s)


... (83 outputs left out)

[93/96] fa/zscore/h3: 18 results +models (3105s)


[94/96] fa/zscore/h4: 18 results +models (3132s)


[95/96] fa/zscore/h5: 18 results +models (3158s)


[96/96] fa/zscore/h6: 18 results +models (3183s)

All combos done in 3183s
Models saved: 1080


In [9]:
# --- Post-training check ---
# Six models per combination (rf, lgbm, xgb, logreg, plus the persistence and
# climatology baselines), three folds, so 18 result rows per combo.
# Fitted artifacts are written for every combination, so the application can
# explain any disease, definition and horizon with SHAP.
import pandas as pd

MODELS_PER_COMBO = 6
FOLDS = 3
ROWS_PER_COMBO = MODELS_PER_COMBO * FOLDS

result_files = sorted(RESULTS_DIR.glob("*.parquet"))
print(f"Combos saved: {len(result_files)} of 96")
if not result_files:
    print("Nothing on disk yet - run the training cell above first.")
else:
    todos = pd.concat([pd.read_parquet(f) for f in result_files], ignore_index=True)
    print(f"Result rows: {len(todos):,}")
    print(f"Models:      {sorted(todos['model'].unique())}")
    print(f"Diseases:    {sorted(todos['disease'].unique())}")
    print(f"Definitions: {sorted(todos['definition'].unique())}")

    por_combo = todos.groupby(["disease", "definition", "h"]).size()
    print(f"\nRows per combo - min {por_combo.min()}, max {por_combo.max()}, "
          f"median {int(por_combo.median())}")
    print(f"({ROWS_PER_COMBO} = {MODELS_PER_COMBO} models x {FOLDS} folds; fewer is"
          f" expected for a rare disease, or a fold with no outbreak to score)")

    dengue = todos[todos["disease"] == "dengue"].groupby(["definition", "h"]).size()
    if len(dengue):
        print(f"\nDengue: min {dengue.min()}, max {dengue.max()} "
              f"(expect {ROWS_PER_COMBO} throughout - dengue has outbreaks in every fold)")

# Feature count. This is the guard against silently training on a stale `feats`:
# 125 is the correct number, 137 means the 12 endemic-channel features that leak
# are still in - restart the kernel and rebuild.
n_feats = len([c for c in feats.columns if c not in ("cod_ibge", "ano", "mes")])
print(f"\nFeatures used: {n_feats}")
if n_feats != 125:
    raise RuntimeError(f"expected 125 features, got {n_feats} - stale `feats`, restart the kernel")

saved = list(MODELS_DIR.glob("*.joblib"))
expected_models = len(result_files) * FOLDS * 4
print(f".joblib saved: {len(saved)} (expected {expected_models} = "
      f"{len(result_files)} combos x {FOLDS} folds x 4 trainable models)")
if saved and len(saved) < expected_models:
    print("  fewer than expected: some combination has no model and cannot be "
          "explained in the application")


Combos saved: 90 of 96
Result rows: 1,620
Models:      ['climatology', 'lgbm', 'logreg', 'persistence', 'rf', 'xgb']
Diseases:    ['chikungunya', 'dengue', 'fa', 'zika']
Definitions: ['canal', 'inc100', 'inc300', 'zscore']

Rows per combo - min 18, max 18, median 18
(18 = 6 models x 3 folds; fewer is expected for a rare disease, or a fold with no outbreak to score)

Dengue: min 18, max 18 (expect 18 throughout - dengue has outbreaks in every fold)

Features used: 125
.joblib saved: 1080 (expected 1080 = 90 combos x 3 folds x 4 trainable models)


## Section 6 — Production models

Section 5 answers *how good is this method*: it holds out 2022, 2023 and 2024 and
never lets a fold see its own future. That is the right design for a claim about
performance, and it is what the paper reports.

It is the wrong design for a forecast. The most recent fold trains on outcomes up
to 2023, so roughly the last eighteen months of collected data are never used by
any model, and the months a manager actually cares about, the ones whose outcome
is not known yet, are never scored at all.

This section fits one model per disease, definition, horizon and model type on
**every row whose outcome is known**, and then scores the rows whose outcome is
not known. Those scores are the forecast the application shows, and the saved
model is what SHAP explains when a user asks why a municipality was flagged.

Two directories are written, both separate from Section 5 so the evaluation and
the forecast can never be confused:

- `models_producao/` — one fitted model per combination, 360 files
- `predictions_producao/` — the score for each municipality-month whose outcome
  is still unknown, plus the month being predicted

These models have no honest test score, by construction: there is no held-out
year left to score them on. Their expected performance is the one Section 5
measured for the same combination.

In [10]:
# ===============================================================
# Section 6 - Production models
#   One fit per (disease, definition, horizon, model) on every labelled row,
#   then a score for the rows whose outcome is not known yet.
#   Roughly 360 fits, about twenty minutes, about 300 MB.
#   Checkpointed the same way as Section 5.
# ===============================================================
RODAR_PRODUCAO = True

PROD_MODELS_DIR = OUT_DIR / "models_producao"
PROD_PRED_DIR   = OUT_DIR / "predictions_producao"
for d in (PROD_MODELS_DIR, PROD_PRED_DIR):
    d.mkdir(parents=True, exist_ok=True)


def prepare_work_producao(disease, definition, h):
    """Like prepare_work, but keeps the rows whose outcome is not known yet.

    Those are the last h months of the series: the month they predict has not
    happened, so build_target leaves their target null. Section 5 drops them;
    here they are exactly what we want to score.
    """
    work = feats.copy()
    work["target"] = build_target(y_labels, disease, definition, h).values
    work = add_target_year(work, h)
    feat_cols = [c for c in work.columns if c not in FEAT_COLS_EXCLUDE]
    return work, feat_cols


def mes_alvo(ano, mes, h):
    """Calendar month being predicted, h months after the observation."""
    total = ano * 12 + (mes - 1) + h
    return total // 12, total % 12 + 1


if not RODAR_PRODUCAO:
    print("RODAR_PRODUCAO is False: skipping.")
else:
    t0 = time.time()
    feitos = pulados = 0
    for i, (disease, definition, h) in enumerate(combos, 1):
        alvo_pred = PROD_PRED_DIR / f"{disease}_{definition}_h{h}.parquet"
        tem_modelos = len(list(PROD_MODELS_DIR.glob(f"{disease}_{definition}_h{h}_*.joblib"))) >= 4
        if alvo_pred.exists() and tem_modelos:
            pulados += 1
            continue

        t1 = time.time()
        work, feat_cols = prepare_work_producao(disease, definition, h)
        conhecido = work["target"].notna()
        tr, futuro = work[conhecido], work[~conhecido]

        # A model needs both classes to be fitted at all. Six of the ninety-six
        # combinations never produce an outbreak anywhere in the series.
        if tr["target"].nunique() < 2:
            print(f"[{i}/{len(combos)}] {disease}/{definition}/h{h}: single class, skip")
            continue

        Xtr = preencher(tr[feat_cols])
        ytr = tr["target"].astype(int)
        Xfut = preencher(futuro[feat_cols])

        linhas = []
        for name, factory in model_factories().items():
            modelo = factory()
            modelo.fit(Xtr, ytr)
            joblib.dump(modelo, PROD_MODELS_DIR / f"{disease}_{definition}_h{h}_{name}.joblib")
            if len(futuro):
                p = futuro[["cod_ibge", "ano", "mes"]].copy()
                alvos = [mes_alvo(a, m, h) for a, m in zip(p["ano"], p["mes"])]
                p["ano_alvo"] = [x[0] for x in alvos]
                p["mes_alvo"] = [x[1] for x in alvos]
                p["disease"] = disease
                p["definition"] = definition
                p["h"] = h
                p["model"] = name
                p["proba"] = modelo.predict_proba(Xfut)[:, 1]
                linhas.append(p)

        if linhas:
            pd.concat(linhas, ignore_index=True).to_parquet(alvo_pred)
        feitos += 1
        ultimo = tr[["ano", "mes"]].iloc[-1]
        print(f"[{i}/{len(combos)}] {disease}/{definition}/h{h}: "
              f"fitted on {len(tr):,} rows through {int(ultimo['ano'])}-{int(ultimo['mes']):02d}, "
              f"scored {len(futuro):,} unlabelled ({time.time()-t1:.0f}s)")

    print(f"\nProduction: {feitos} combinations fitted, {pulados} already on disk, "
          f"{(time.time()-t0)/60:.1f} min")
    # A matriz de features vai para disco junto com os modelos. A aplicação
    # precisa dela para explicar uma previsão por SHAP, e reconstruí-la a cada
    # sessão levaria cerca de trinta segundos. Gravada aqui, e não numa célula
    # de feature, porque é artefato de produção e não entra no caminho que o
    # 04_results reexecuta.
    feats.to_parquet(OUT_DIR / "features.parquet", compression="zstd", index=False)
    print(f"  features.parquet:     {feats.shape[0]:,} x {feats.shape[1]}")
    print(f"  models_producao:      {len(list(PROD_MODELS_DIR.glob('*.joblib')))} files")
    print(f"  predictions_producao: {len(list(PROD_PRED_DIR.glob('*.parquet')))} files")


[1/96] dengue/inc100/h1: fitted on 88,365 rows through 2026-05, scored 645 unlabelled (22s)


[2/96] dengue/inc100/h2: fitted on 87,720 rows through 2026-04, scored 1,290 unlabelled (23s)


[3/96] dengue/inc100/h3: fitted on 87,075 rows through 2026-03, scored 1,935 unlabelled (23s)


[4/96] dengue/inc100/h4: fitted on 86,430 rows through 2026-02, scored 2,580 unlabelled (23s)


[5/96] dengue/inc100/h5: fitted on 85,785 rows through 2026-01, scored 3,225 unlabelled (22s)


[6/96] dengue/inc100/h6: fitted on 85,140 rows through 2025-12, scored 3,870 unlabelled (22s)


... (83 outputs left out)

[93/96] fa/zscore/h3: fitted on 87,075 rows through 2026-03, scored 1,935 unlabelled (12s)


[94/96] fa/zscore/h4: fitted on 86,430 rows through 2026-02, scored 2,580 unlabelled (12s)


[95/96] fa/zscore/h5: fitted on 85,785 rows through 2026-01, scored 3,225 unlabelled (11s)


[96/96] fa/zscore/h6: fitted on 85,140 rows through 2025-12, scored 3,870 unlabelled (11s)

Production: 90 combinations fitted, 0 already on disk, 27.0 min
  models_producao:      360 files
  predictions_producao: 90 files


## Section 7 — Operating thresholds per horizon

A score of 0.50 is not the same decision at one month and at six. The score
shrinks as the forecast moves away: at one month ahead, 0.50 catches 78.1% of
outbreaks; at six months, the same cut catches 56.9%, not because the model
knows less about which municipalities are at risk, but because it is less
confident about all of them at once. Ordering survives the horizon; absolute
magnitude does not.

So this section finds, for each disease, definition, model and horizon, the
score that maximises F1, and writes it as the operating point the application
uses instead of a fixed 0.50.

**Why not calibrate the score into a probability.** That is the obvious fix and
it does not work here. The outbreak rate moves enormously between years, and a
calibration map learned on one year is wrong on the next: a mapping fitted when
one municipality-month in six is an outbreak cannot hold in a year when it is
nearly one in two. Next year's base rate is not knowable before next year. What
does transfer is the ordering, which is what AUROC measures and what does not
depend on the base rate at all. The platform therefore presents a risk score and
a position in a queue, never a chance, and uses the threshold only as an
operating point.

**Why the median across folds and not one pooled fit.** Each test year has its
own optimum, and pooling would let the largest epidemic year decide the
threshold for the ordinary ones. The median is the conservative aggregate.

**Precision and recall without looking at the answer.** The median cut is
chosen on the same test years it is then scored on, which flatters it. So each
test year is also scored with the cut chosen on the other test years alone, and
that precision and recall are written next to the in-sample ones; they are the
numbers the application shows. When no other year has enough outbreaks to
choose a cut (zika under the historical-range definition has 14 and 15 outbreak
months in 2022 and 2023), that year is scored at 0.50.

**Which combinations the application offers.** A disease, definition, horizon
and model is offered only if the model separates outbreak months from calm ones
reasonably well in the test years: mean AUROC of at least 0.70, the usual floor
for acceptable discrimination. Below it the model is ranking municipalities
barely better than chance, and an alert built on that ranking would cost a
health department more in false alarms than it gives back. The column `oferecer`
records the decision, and the application reads it.

With this floor, twelve disease-definition-horizon combinations have no model
left and disappear: dengue under the historical-range and standard-score
definitions at four to six months, and chikungunya under the two absolute
definitions at the longer horizons. The default model of each disease passes at
every horizon.

In [ ]:
# ===============================================================
# Section 7 - Operating thresholds
#   For each (disease, definition, model, horizon): the F1-maximising score,
#   computed inside each fold and aggregated by the median across folds.
#   Reads the out-of-sample predictions Section 5 wrote; costs seconds.
# ===============================================================
RODAR_LIMIARES = True

from sklearn.metrics import f1_score, precision_score, recall_score

LIMIARES_ARQ = OUT_DIR / "limiares_operacao.parquet"
GRADE = np.round(np.arange(0.05, 0.96, 0.05), 2)
MIN_POSITIVOS = 20      # abaixo disso o ótimo é ruído, então fica o padrão
LIMIAR_PADRAO = 0.50
AUROC_MINIMO = 0.70     # abaixo disso o modelo não entra no aplicativo


def melhor_limiar(y, p):
    """Ponto da grade que maximiza F1, com o F1 obtido."""
    escores = [f1_score(y, (p >= t).astype(int), zero_division=0) for t in GRADE]
    i = int(np.argmax(escores))
    return float(GRADE[i]), float(escores[i])


if not RODAR_LIMIARES:
    print("RODAR_LIMIARES is False: skipping.")
else:
    linhas = []
    for arq in sorted(PRED_DIR.glob("*.parquet")):
        preds = pd.read_parquet(arq)
        for (disease, definition, h, modelo), g in preds.groupby(
                ["disease", "definition", "h", "model"], sort=False):
            # Baselines have no threshold to tune: persistence is already 0/1 and
            # climatology is a historical rate, not a model score.
            if modelo in ("persistence", "climatology"):
                continue
            por_fold = {}
            for ty, gf in g.groupby("test_year"):
                if gf["y_true"].sum() < MIN_POSITIVOS or gf["y_true"].nunique() < 2:
                    continue
                por_fold[ty] = melhor_limiar(gf["y_true"].values, gf["proba"].values)
            if not por_fold:
                continue
            ts = np.array([t for t, _ in por_fold.values()])
            limiar = float(np.median(ts))
            y, p = g["y_true"].values, g["proba"].values
            marca = p >= limiar

            # Out-of-year scoring: each test year is marked with the cut chosen
            # on the other test years only, or 0.50 when none of them qualifies.
            y_fora, marca_fora = [], []
            for ty, gf in g.groupby("test_year"):
                outros = [t for k, (t, _) in por_fold.items() if k != ty]
                corte = float(np.median(outros)) if outros else LIMIAR_PADRAO
                y_fora.append(gf["y_true"].values)
                marca_fora.append((gf["proba"].values >= corte).astype(int))
            y_fora, marca_fora = np.concatenate(y_fora), np.concatenate(marca_fora)
            linhas.append({
                "disease": disease, "definition": definition, "h": int(h),
                "model": modelo, "limiar": limiar,
                "limiar_min": float(ts.min()), "limiar_max": float(ts.max()),
                "folds": len(por_fold),
                "f1_no_limiar": f1_score(y, marca.astype(int), zero_division=0),
                "f1_em_050": f1_score(y, (p >= 0.5).astype(int), zero_division=0),
                "recall_no_limiar": recall_score(y, marca.astype(int), zero_division=0),
                "recall_em_050": recall_score(y, (p >= 0.5).astype(int), zero_division=0),
                "precisao_no_limiar": precision_score(y, marca.astype(int), zero_division=0),
                "recall_fora_do_ano": recall_score(y_fora, marca_fora, zero_division=0),
                "precisao_fora_do_ano": precision_score(y_fora, marca_fora, zero_division=0),
            })

    limiares = pd.DataFrame(linhas)

    # Mean AUROC and AUPRC over the test years, the same numbers Section 5
    # reports, the outbreak rate that AUPRC is read against, and the decision
    # to offer the combination in the application or not.
    chave = ["disease", "definition", "h", "model"]
    medias = (pd.concat([pd.read_parquet(f) for f in RESULTS_DIR.glob("*.parquet")])
                .groupby(chave)
                .agg(auroc=("auroc", "mean"), auprc=("auprc", "mean"),
                     n_pos=("n_pos", "sum"), n_test=("n_test", "sum"))
                .reset_index())
    medias["taxa_surto"] = medias["n_pos"] / medias["n_test"]
    limiares = limiares.merge(medias, on=chave, how="left")
    limiares["oferecer"] = limiares["auroc"] >= AUROC_MINIMO
    limiares.to_parquet(LIMIARES_ARQ, index=False)
    print(f"Operating thresholds: {len(limiares)} combinations -> {LIMIARES_ARQ.name}")
    n_comb = limiares.groupby(["disease", "definition", "h"])["oferecer"].any()
    print(f"  offered in the application (AUROC >= {AUROC_MINIMO}): "
          f"{int(limiares['oferecer'].sum())} of {len(limiares)} disease-definition-"
          f"horizon-model; {int(n_comb.sum())} of {len(n_comb)} disease-definition-horizon")

    if len(limiares):
        ganho = limiares["f1_no_limiar"] - limiares["f1_em_050"]
        print(f"  F1 gain over a fixed 0.50: median {ganho.median():+.3f}, "
              f"worst {ganho.min():+.3f}, best {ganho.max():+.3f}")
        print(f"  threshold spread across folds: median {(limiares.limiar_max - limiares.limiar_min).median():.2f}")
        otimismo = limiares["recall_no_limiar"] - limiares["recall_fora_do_ano"]
        print(f"  recall lost when the cut is chosen on the other years: "
              f"median {otimismo.median():.3f}, worst {otimismo.max():.3f}")
        print("\n  Dengue, incidence definition, random forest, by horizon:")
        d = limiares[(limiares.disease == "dengue") & (limiares.definition == "inc100")
                     & (limiares.model == "rf")].sort_values("h")
        for _, r in d.iterrows():
            print(f"    h={int(r.h)}  limiar {r.limiar:.2f}  "
                  f"recall {100*r.recall_em_050:.1f}% -> {100*r.recall_no_limiar:.1f}%  "
                  f"F1 {r.f1_em_050:.3f} -> {r.f1_no_limiar:.3f}")
